#### get text content from data
If you use this dataset, please cite the following paper:
Urchs et al. (2025): taz2024full: Analysing German Newspapers for Gender Bias and Discrimination across Decades, Findings of ACL 2025.

format:
- meta data:

    - "published_on": publishing date in the format YYYY-MM-DDThh:mm:ss+01:00

    - "contains_actors": boolean indicating whether person entities were detected

    - "crawled_on": crawling date

    - "language": always "de"

    - "type": always "article"

    - "author": article author

    - "keywords": topic keywords

    - "token_count": number of tokens


- text:

    - "title": article title

    - "teaser": short summary

    - "text": main body of the article

In [ ]:
import pandas as pd
import ast
import json
from pprint import pprint
import os
import re
from io import StringIO
from tqdm import tqdm
import string
from sklearn.feature_extraction.text import TfidfVectorizer

import spacy
nlp = spacy.load('de_core_news_lg') # oder de_core_news_sm
def flatten(xss):
    return [x for xs in xss for x in xs]

In [22]:
c_directory = r"corpus" 

# Iterate over files in directory
dict_year = {}
for name in os.listdir(c_directory):
    dir = os.path.join(c_directory, name)
    for n in os.listdir(dir):
        with open(os.path.join(dir, n), encoding="utf-8") as f:
            data = json.load(f)
            dict_year[name] = data
print(dict_year.keys())

dict_keys(['2013', '2014', '2022', '2024', '2023', '2015', '2012', '1988', '1986', '1987', '1980', '1989', '1993', '1994', '1995', '1992', '2008', '2001', '2006', '2007', '2000', '2009', '2017', '2010', '2019', '2021', '2020', '2018', '2011', '2016', '1985', '1983', '1984', '1997', '1990', '1999', '1998', '1991', '1996', '2005', '2002', '2003', '2004'])


In [23]:
# separate sentencesy with spacy
def separate_sentences(sents):
    doc = nlp(sents)
    res = []
    for sent in doc.sents:
     res.append(sent.text.lower())
     # entfernen von vorkommen von "h hb hh ki", tags?
    return res

In [24]:
# keep text content and id only, divide text into sentences
def save_sentences_to_file():
    text_dir = "corpus_texts/"
    # Create the directory
    try:
        os.mkdir(text_dir)
        print(f"Directory '{text_dir}' created successfully.")
    except FileExistsError:
        print(f"Directory '{text_dir}' already exists.")

    #fill dir
    for key in tqdm(dict_year.keys()):
        text_dict = {}
        for k in dict_year[key].keys():
            text = dict_year[key][k]['text']['text']
            text_dict[k] = text
            with open(f"{text_dir}{k}.txt", "w") as f:
                    f.write(json.dumps(separate_sentences(str(text))))

# keep text content and id only
def save_txt_to_file():
    text_dir = "corpus_texts/"
    # Create the directory
    try:
        os.mkdir(text_dir)
        print(f"Directory '{text_dir}' created successfully.")
    except FileExistsError:
        print(f"Directory '{text_dir}' already exists.")

    #fill dir
    for key in tqdm(dict_year.keys()):
        text_dict = {}
        for k in dict_year[key].keys():
            text = dict_year[key][k]['text']['text']
            text_dict[k] = text
            with open(f"{text_dir}{k}_full.txt", "w") as f:
                    f.write(str(text))

#save_sentences_to_file()
#save_txt_to_file()

#### get sentences with occurences of AI related words
Notes: 
- Englisch and German for certain terms?
- Specific product names
- ML/AI system terminology
- Nouns only
- Abreviations, too



steps:
- build AI vocabulary
- filter by sentence containing at least one AI vocab
- multiply sentences if they contain multiple terms 

In [25]:
# AI-terms from ANTHROSCORE paper, transalted with deepL
ai_terms = list(ast.literal_eval(open("config_files/ger_anthroscore_KI_terms.txt").read()).values())
regex_ai = "".join(["\\b"+x.replace(" ","\s")+"\\b|" for x in ai_terms])[:-1]

# get AI-related words, sorted according to 'Häufigkeitsklassen'
# https://corpora.uni-leipzig.de/de/res?corpusId=deu_news_2024&word=KI
# TODO: Add syntax so that ANTHROSCORE recognmizes n-grams 
ki_terms = ['KI','Künstliche Intelligenz', 'ChatGPT', 'AI', 'Copilot', 'Gemini', 'Chatbot','LLM', 'Chat-GPT',
            'Sprachmodell', 'KI-Modell', 'KI-System', 'KI-Tool', 'Artificial Intelligence', 'KI-Anwendung'] + ai_terms
ki_terms = [x.lower() for x in ki_terms]
regex_ki = "".join(["\\b"+x.replace(" ","\s")+"\\b|" for x in ki_terms])[:-1]
print(regex_ki)

\bki\b|\bkünstliche\sintelligenz\b|\bchatgpt\b|\bai\b|\bcopilot\b|\bgemini\b|\bchatbot\b|\bllm\b|\bchat-gpt\b|\bsprachmodell\b|\bki-modell\b|\bki-system\b|\bki-tool\b|\bartificial\sintelligence\b|\bki-anwendung\b|\bpalm\b|\blms\b|\bllama\b|\btransformer\b|\bsprachmodelle\b|\bsprachmodell\b|\bgpt\b|\bplms\b|\bvortrainierte\ssprachmodelle\b|\bgpt-2\b|\bxlnet\b|\bgroße\ssprachmodelle\b|\bllms\b|\bgpt-3\b|\bstiftungsmodell\b|\bgpt-neo\b|\bgpt-j\b|\bchatgpt\b|\bgpt-4\b


<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:11: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:11: SyntaxWarning: invalid escape sequence '\s'
/var/folders/0h/lr036jk55nq7b5pmf0kg0qq40000gn/T/ipykernel_2162/1902452717.py:3: SyntaxWarning: invalid escape sequence '\s'
  regex_ai = "".join(["\\b"+x.replace(" ","\s")+"\\b|" for x in ai_terms])[:-1]
/var/folders/0h/lr036jk55nq7b5pmf0kg0qq40000gn/T/ipykernel_2162/1902452717.py:11: SyntaxWarning: invalid escape sequence '\s'
  regex_ki = "".join(["\\b"+x.replace(" ","\s")+"\\b|" for x in ki_terms])[:-1]


#### Sätze säubern, filtern
- Satzzeichen
- (Html, etc)
- POS (use spacy bc TIGER corpus trained)
- Lexikon für Verb- / Adjektivtypen

In [26]:
# preprocessing auxiliary functions

stopwords = open("config_files/spacy_stopwords.txt").read()

# this one is not needed if working on Sentence_spacy
def remove_punctuation(corpus):
    cleaned_corpus = corpus.translate(str.maketrans('', '', string.punctuation))
    return(cleaned_corpus)

# remove stop words from text
def remove_stop_words_from_text(cleaned_corpus):
    out = [word for word in cleaned_corpus if not word in stopwords]
    out_text = ' '.join(out)
    return out_text  # , out


def remove_stop_words(token_list):
    return [token for token in token_list if not token.is_stop]

    
def lemmatizing(token_list):
    return [token.lemma_ for token in token_list]
    
def get_POS(token_list):
    return [token.pos_ for token in token_list]

In [27]:
# use tf-idf to get keywords from texts
# corpus a list of sentences/documents
def calc_tf_idf(corpus):
    vectorizer = TfidfVectorizer(ngram_range=(1,2))
    tfidf_m = vectorizer.fit_transform(corpus)
    res = vectorizer.get_feature_names_out()
    #print(tfidf_m.shape)
    return res #array of list(s?)

#### get sentences with ki_terms

In [34]:
# filter ki_terms
def filter_full_ki_terms_regex(c_dir,regex,path):
    ki_sentences = {}
    try:
        for dir in tqdm(os.listdir(c_dir)):
            sent_l = []
            if dir.endswith("_full.txt"):
                sent_l = open(os.path.join(c_dir,dir)).read()
                # could also account for lemmata, etc
                sent_l = remove_punctuation(sent_l)        
                # filter
                res_tmp = []
                if re.search(regex, sent_l):
                    #print("res",sent, re.search(regex_ki, sent).string)
                    res_tmp.append(sent_l)
                # add to dict
                if len(res_tmp) > 0:
                    ki_sentences [dir] =  res_tmp
    except:
        with open(c_dir) as f:
            d = json.load(f)

        for elem in d:
            sent = remove_punctuation(elem["translation"])
            # filter
            res_tmp = []
            if re.search(regex, sent):
                #print("res",sent, re.search(regex_ki, sent).string)
                res_tmp.append(sent)
            # add to dict
            if len(res_tmp) > 0:
                ki_sentences [elem["id"]] =  res_tmp

    print(len(ki_sentences))
    #pprint(filtered_sentences_json)
    # save to file
    with open(f'generated_data/{path}', 'w') as f:
        json.dump(ki_sentences, f)
    
    
def filter_ki_terms_regex(c_dir,regex,path):
    ki_sentences = {}
    try:
        for dir in tqdm(os.listdir(c_dir)):
            if not dir.endswith("_full.txt"):
                with open(os.path.join(c_dir,dir)) as f:
                    sent_l = json.load(f)
                    # could also account for lemmata, etc
                    sent_l = [remove_punctuation(sent) for sent in sent_l]   
                #filter
                res_tmp = []
                for sent in sent_l:
                    if re.search(regex, sent):
                    #print("res",sent, re.search(regex_ki, sent).st‚ring)
                        res_tmp.append(sent)
                    # add to dict
                    if len(res_tmp) > 0:
                        ki_sentences [dir] =  res_tmp
    except:
        with open(c_dir) as f:
            d = json.load(f)

        for elem in d:
            sent = remove_punctuation(elem["translation"])
            #filter
            res_tmp = []
            if re.search(regex, sent):
            #print("res",sent, re.search(regex_ki, sent).st‚ring)
                res_tmp.append(sent)
            # add to dict
            if len(res_tmp) > 0:
                ki_sentences [elem["id"]] =  res_tmp
            
    print(len(ki_sentences))
    #pprint(filtered_sentences_json)
    # save to file
    with open(f'generated_data/{path}', 'w') as f:
        json.dump(ki_sentences, f)

In [46]:
# filter using TF-IDF instead of all tokens
# function filter_full_ki_terms_tfidf() following not useful since tfidf needs sentence list as corpus

def filter_ki_terms_tfidf(c_dir,terms,path):
    ki_sentences = {}
    try: 
        for dir in tqdm(os.listdir(c_dir)):
            if not dir.endswith("_full.txt"):
                #sent_l = []
                with open(os.path.join(c_dir,dir)) as f:
                    sent_l = json.load(f)
                    # could also account for lemmata, etc
                    sent_l = [remove_punctuation(sent) for sent in sent_l]
                # compare tf-idf with ki_terms here
                if len(sent_l) > 1:
                    tfidf_words = calc_tf_idf(sent_l)
                    if any(tfidf_words) in terms:
                        ki_sentences[dir] = {"sentences": str(sent_l), "top_words": str(tfidf_words)}
                    else:
                        ki_sentences[dir] = {"sentences":None, "top_words": str(tfidf_words)}
                else:
                    continue
                    #print(sent_l)
    except:
        with open(c_dir) as f:
            d = json.load(f)

        for elem in d:
            sent = [remove_punctuation(elem["translation"])]
            if len(sent[0].split()) > 1:
                tfidf_words = calc_tf_idf(sent)
                if any(tfidf_words) in terms:
                    ki_sentences[elem["id"]] = {"sentences": elem["text"], "top_words": str(tfidf_words)}
                else:
                    ki_sentences[elem["id"]] = {"sentences": None, "top_words": str(tfidf_words)}
            else:
                continue

    print(len(ki_sentences))
    #pprint(filtered_sentences_json)
    # save to file
    with open(f'generated_data/{path}', 'w') as f:
        json.dump(ki_sentences, f)

In [49]:
c_txt_dir = r"corpus_texts" 

ki_sentences_json = filter_ki_terms_regex(c_txt_dir, regex_ki,"KI_sentences_full.json")
ki_sentences_full_json =filter_full_ki_terms_regex(c_txt_dir, regex_ki, "KI_sentences.json")
ki_sentences_tfidf_json = filter_ki_terms_tfidf(c_txt_dir, ki_terms, "KI_sentences_tfidf.json")

100%|██████████| 306612/306612 [01:01<00:00, 4977.69it/s]


869


100%|██████████| 306612/306612 [01:01<00:00, 4947.44it/s]


150


100%|██████████| 306612/306612 [03:26<00:00, 1483.36it/s]


152225


##### get AI terms of translations

In [48]:
c_txt_dir_full_eng ="generated_data/KI_sentences_full_translated.json"
c_txt_dir_eng ="generated_data/KI_sentences_translated.json"

ai_sentences_json = filter_ki_terms_regex(c_txt_dir_eng, regex_ai, "AI_sentences_full.json")
ai_sentences_full_json = filter_full_ki_terms_regex(c_txt_dir_full_eng, regex_ai, "AI_sentences.json")
ai_sentences_tfidf_json = filter_ki_terms_tfidf(c_txt_dir_eng, ai_terms, "AI_sentences_tfidf.json")


75
2
574


### Daten veranschaulichen:
- Häufigkeit der KI Terme
- Häufigste Wörter
- length of sentences / number of clauses
- number / type of verbs
- number / type of adjektives

In [50]:
plots_dir = "plots/"
docs_df = pd.read_json(StringIO(open('generated_data/KI_sentences_tfidf.json').read()), orient="index")
docs_df.head()

,sentences,top_words
6036703.txt,NaN,['13' '13 prozent' '1932' '1932 entmachtete' '...
0224025.txt,NaN,['am' 'am wochenende' 'an' 'an der' 'angegriff...
5903496.txt,NaN,['19' '19 jahrhundert' '1900' '1900 brachte' '...
1184664.txt,NaN,['1989' '70' '70 überschritten' '74' '74 ein' ...
806049.txt,NaN,['750' '750 gramm' 'auf' 'auf krankenschein' '...


In [ ]:
# Häufigste Wörter insgesamt
def top_words(df_col, k):
    top_k = df_col.value_counts().head(k)
    print(f"\n{k} most frequent words:\n", top_k)
    return top_k

# Häufigste Wörter pro jahr 
def top_words_year(df, k):
    # TODO:find year, jahr aus id -> datei -> metadaten

    # group by year
    years = df.groupby(['_'])#.mean()
    # calc top words
    top_y = [(y[0],top_words(y[1],k)) for y in years]
    for y in top_y:
        print(f"\n{k} most frequent words:\n", y)
    return top_y

# Typ der Wörter

#### TFIDF of KI_sentences filtered with regex

In [ ]:
# compare KI_sentences and KI_sentences_full ids

# combine

# calc tfidf on result

# visualize, ideally against other tfidf(s)


#### KI_sentence stats

In [ ]:
docs_ki = docs_df[docs_df["sentences"].notna()]
docs_ki.head()

In [ ]:
docs_sents = pd.read_json(StringIO(open('generated_data/KI_sentences.json').read()), orient="index")
docs_full = pd.read_json(StringIO(open('generated_data/KI_sentences_full.json').read()), orient="index")

docs_sents.head()

In [ ]:
# Häufigste Wörter pro satz / text

# Häufigkeit der einzelnen KI Terme

# average length of sentences / number of clauses

In [ ]:
# number and type of verbs

# number and type of adjektives